In [ ]:
%config InlineBackend.figure_formats = ['svg']
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import sparse
import scipy.sparse.linalg as spla
import pyamg
from functools import reduce
import itertools
from tabulate import tabulate
import IPython.display as display

In [ ]:
plt.style.use("dark_background")
# mpl.rcParams["figure.facecolor"] = 'none'
mpl.rcParams["axes.facecolor"] = "none"
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["figure.figsize"] = [12.8, 9.6]
mpl.rcParams["axes3d.xaxis.panecolor"] = (0.95, 0.95, 0.95, 0.125)
mpl.rcParams["axes3d.yaxis.panecolor"] = (0.9, 0.9, 0.9, 0.125)
mpl.rcParams["axes3d.zaxis.panecolor"] = (0.925, 0.925, 0.925, 0.125)
mpl.rcParams["grid.alpha"] = 1.0
mpl.rcParams["grid.color"] = (1.0, 1.0, 1.0, 0.125)
mpl.rcParams["grid.linestyle"] = "-"
mpl.rcParams["grid.linewidth"] = 0.5
mpl.rcParams

# Notes

## Library

Spec for mixed BC:

$$\begin{align*}
    -\triangle u &= f,            \qquad x \in \Omega \\
    \alpha u + \partial_n u &= g, \qquad x \in \partial \Omega
\end{align*}$$

In [ ]:
from collections import namedtuple

Dimension = namedtuple(
    "Dimension", ["low", "high", "num_divs", "low_robin", "high_robin"]
)
BoxDomain = namedtuple("BoxDomain", ["dimensions"])

In [ ]:
def fdm_neg_lapl_1d_mat(dim: Dimension):
    interior_size = dim.num_divs - 1
    full_size = (
        interior_size + int(dim.low_robin is not None) + int(dim.high_robin is not None)
    )
    dim_len = dim.high - dim.low
    h = dim_len / dim.num_divs
    invhsqr = 1 / (h * h)

    return (
        sparse.block_array(
            (
                [
                    [
                        sparse.csr_array(
                            ([2.0 * (1 + h * dim.low_robin), -2.0], [0, 1], [0, 2]),
                            shape=(1, full_size),
                        )
                    ]
                ]
                if dim.low_robin is not None
                else [[None]]
            )
            + (
                [
                    [
                        sparse.diags_array(
                            [2.0, -1, -1],
                            offsets=np.array([0, 1, -1])
                            + int(dim.low_robin is not None),
                            shape=(interior_size, full_size),
                        )
                    ]
                ]
            )
            + (
                [
                    [
                        sparse.csr_array(
                            (
                                [2.0 * (1 + h * dim.high_robin), -2.0],
                                [full_size - 1, full_size - 2],
                                [0, 2],
                            ),
                            shape=(1, full_size),
                        )
                    ]
                ]
                if dim.high_robin is not None
                else [[None]]
            )
        )
        * invhsqr
    )


def smallest_hsqr(dom: BoxDomain):
    return (h := min((dim.high - dim.low) / dim.num_divs for dim in dom.dimensions)) * h


def fdm_neg_lapl_unif_disc_mat(dom: BoxDomain):
    return reduce(
        sparse.kronsum,
        (fdm_neg_lapl_1d_mat(dim) for dim in reversed(dom.dimensions)),
    )

In [ ]:
def make_coordinate_grids(dom: BoxDomain):
    return np.meshgrid(
        *(
            np.linspace(dim.low, dim.high, num=dim.num_divs + 1)[
                slice(
                    1 if dim.low_robin is None else 0,
                    -1 if dim.high_robin is None else None,
                )
            ]
            for dim in dom.dimensions
        ),
        indexing="ij"
    )

In [ ]:
def calc_forcing_term(
    forcing_fn, coord_grids, ghost_term_bd=True, domain: BoxDomain = None
):
    forcing_term = forcing_fn(coord_grids)
    if not ghost_term_bd:
        num_dims = len(domain.dimensions)
        for i, dim in enumerate(domain.dimensions):
            for face_idx, robin in [
                (0, dim.low_robin),
                (-1, dim.high_robin),
            ]:
                idx_selector = tuple(
                    face_idx if i == j else slice(None) for j in range(num_dims)
                )
                if robin is not None:
                    # Robin
                    forcing_term[idx_selector] = 0.0

    return forcing_term


def manufactured_boundary_terms(dom: BoxDomain, coords, soln, grad):
    bd_terms = np.zeros_like(coords[0])
    num_dims = len(dom.dimensions)
    for i, dim in enumerate(dom.dimensions):
        dim: Dimension = dim
        dim_len = dim.high - dim.low
        h = dim_len / dim.num_divs
        invh = 1.0 / h
        invhsqr = invh * invh

        for face_idx, robin, bd_coord, normal_sign in [
            (0, dim.low_robin, dim.low, -1.0),
            (-1, dim.high_robin, dim.high, 1.0),
        ]:
            idx_selector = tuple(
                face_idx if i == j else slice(None) for j in range(num_dims)
            )
            face_coords = tuple(
                bd_coord if i == j else grid[idx_selector]
                for j, grid in enumerate(coords)
            )
            if robin is None:
                # Dirichlet
                bd_terms[idx_selector] += invhsqr * soln(face_coords)
            else:
                # Robin
                bd_terms[idx_selector] += (
                    2.0
                    * invh
                    * (robin * soln(face_coords) + normal_sign * grad(face_coords)[i])
                )

    return bd_terms

In [ ]:
def get_interior_selector(domain: BoxDomain):
    return tuple(
        slice(
            1 if dim.low_robin is None else 0,
            -1 if dim.high_robin is None else None,
        )
        for dim in domain.dimensions
    )


def get_bd_values(field_values, domain: BoxDomain):
    int_selector = get_interior_selector(domain)
    mask = np.full_like(field_values, False)
    mask[int_selector] = True
    return np.ma.MaskedArray(field_values, mask)


def interior_rms_norm(field_values, domain: BoxDomain):
    int_selector = get_interior_selector(domain)
    return np.sqrt(np.mean(np.square(field_values[int_selector])))


def interior_linf_norm(field_values, domain: BoxDomain):
    int_selector = get_interior_selector(domain)
    return np.max(np.abs(field_values[int_selector]))


def bd_rms_norm(field_values, domain: BoxDomain):
    bd_values = get_bd_values(field_values, domain)
    return np.sqrt(np.mean(np.square(bd_values)))


def bd_linf_norm(field_values, domain: BoxDomain):
    bd_values = get_bd_values(field_values, domain)
    return np.max(np.abs(bd_values))

In [ ]:
def do_a_quickie():
    def exact_soln(coords):
        x, y = coords
        return np.sin(x + 1.0) * np.sin(y + 2.0)

    def exact_grad(coords):
        x, y = coords
        dxg = np.cos(x + 1.0) * np.sin(y + 2.0)
        dyg = np.sin(x + 1.0) * np.cos(y + 2.0)
        return np.stack([dxg, dyg])

    def forcing(coords):
        return 2.0 * exact_soln(coords)

    domain = BoxDomain(
        [
            Dimension(0.0, 1.0, 256, None, 0.0),
            Dimension(0.0, 1.0, 128, 0.0, None),
        ]
    )

    hsqr = smallest_hsqr(domain)
    coords = make_coordinate_grids(domain)
    poisson_lhs = hsqr * fdm_neg_lapl_unif_disc_mat(domain)

    poisson_rhs = (
        hsqr
        * (
            calc_forcing_term(forcing, coords, True, domain)
            + manufactured_boundary_terms(domain, coords, exact_soln, exact_grad)
        ).ravel()
    )

    # approx = spla.spsolve(poisson_lhs, poisson_rhs)
    # residual = np.max(np.abs(poisson_lhs @ approx - poisson_rhs))
    # error = np.max(np.abs(exact_soln(coords) - approx.reshape(coords[0].shape)))
    # residual, error

    ml = pyamg.ruge_stuben_solver(poisson_lhs)
    approx = ml.solve(poisson_rhs, tol=np.finfo(np.float64).eps)
    residual = np.max(np.abs(poisson_lhs @ approx - poisson_rhs))
    error = exact_soln(coords) - approx.reshape(coords[0].shape)
    return residual, [
        norm(error, domain)
        for norm in [interior_rms_norm, interior_linf_norm, bd_rms_norm, bd_linf_norm]
    ]


do_a_quickie()

## Given reproducible model

This model:

$$\begin{align*}
    -\triangle u &= f,            &\qquad (x, y) \in \Omega = (0, 1)^2 \\
    \alpha u + \partial_n u &= r_\alpha, &\qquad (x, y) \in \Gamma_B = \{1\}_x \times (0, 1)_y \\
    u &= g, &\qquad (x, y) \in \Gamma_D = \partial \Omega \setminus \Gamma_B \\
\end{align*}$$

Then $f, r_\alpha, g$ chosen so that $u(x, y) = e^{x + y}.$ Also $\alpha\in\{0,2\}$ is considered.

In [ ]:
def u_exact(coords):
    x, y = coords
    return np.exp(x + y)


def u_exact_grad(coords):
    dxg = u_exact(coords)
    return np.stack([dxg, dxg])


def forcing(coords):
    return -2.0 * u_exact(coords)


domains = [
    BoxDomain(
        [
            Dimension(0.0, 1.0, num_divs, None, robin),
            Dimension(0.0, 1.0, num_divs, None, None),
        ]
    )
    for robin, num_divs in itertools.product((0.0, 2.0), 4 << (np.arange(4) << 1))
]

results = []

for ghost_term_bd, domain in itertools.product((False, True), domains):
    hsqr = smallest_hsqr(domain)
    coords = make_coordinate_grids(domain)
    poisson_lhs = hsqr * fdm_neg_lapl_unif_disc_mat(domain)

    poisson_rhs = (
        hsqr
        * (
            calc_forcing_term(forcing, coords, ghost_term_bd, domain)
            + manufactured_boundary_terms(domain, coords, u_exact, u_exact_grad)
        ).ravel()
    )

    # approx = spla.spsolve(poisson_lhs, poisson_rhs)
    ml = pyamg.ruge_stuben_solver(poisson_lhs)
    approx = ml.solve(poisson_rhs, tol=np.finfo(np.float64).eps)
    residual = (poisson_rhs - poisson_lhs @ approx).reshape(coords[0].shape)
    approx = approx.reshape(coords[0].shape)
    error = u_exact(coords) - approx
    int_rms_rsdl = interior_rms_norm(residual, domain)
    int_linf_rsdl = interior_rms_norm(residual, domain)
    bd_rms_rsdl = bd_rms_norm(residual, domain)
    bd_linf_rsdl = bd_linf_norm(residual, domain)
    int_rms_err = interior_rms_norm(error, domain)
    int_linf_err = interior_linf_norm(error, domain)
    bd_rms_err = bd_rms_norm(error, domain)
    bd_linf_err = bd_linf_norm(error, domain)

    results.append(
        dict(
            ghost_term_bd=ghost_term_bd,
            domain=domain,
            coords=coords,
            hsqr=hsqr,
            poisson_lhs=poisson_lhs,
            poisson_rhs=poisson_rhs,
            approx=approx,
            residual=residual,
            error=error,
            int_rms_rsdl=int_rms_rsdl,
            int_linf_rsdl=int_linf_rsdl,
            bd_rms_rsdl=bd_rms_rsdl,
            bd_linf_rsdl=bd_linf_rsdl,
            int_rms_err=int_rms_err,
            int_linf_err=int_linf_err,
            bd_rms_err=bd_rms_err,
            bd_linf_err=bd_linf_err,
        )
    )

In [ ]:
display.HTML(
    tabulate(
        [
            [
                res["ghost_term_bd"],
                (dim := res["domain"].dimensions[0]).num_divs,
                dim.high_robin,
                res["int_rms_rsdl"],
                res["int_linf_rsdl"],
                res["bd_rms_rsdl"],
                res["bd_linf_rsdl"],
                res["int_rms_err"],
                res["int_linf_err"],
                res["bd_rms_err"],
                res["bd_linf_err"],
            ]
            for res in results
        ],
        headers=[
            'ghost',
            'divs',
            'robin',
            'int RMS rsdl',
            'int Linf rsdl',
            'bd RMS rsdl',
            'bd Linf rsdl',
            'int RMS err',
            'int Linf err',
            'bd RMS err',
            'bd Linf err',
        ],
        tablefmt="html",
    )
)

In [ ]:
fig, ax = plt.subplots()
ax: plt.Axes = ax

ax_im = ax.matshow(
    results[0]["poisson_lhs"].toarray(),
    norm=mpl.colors.CenteredNorm(),
    cmap="berlin",
)
fig.colorbar(ax_im)
plt.show()

print(
    f"symmetricity measure is {spla.norm(poisson_lhs - poisson_lhs.T) / spla.norm(poisson_lhs):0.4f}"
)

In [ ]:
result = results[0]
domain = result["domain"]
xlim = domain.dimensions[0][:2]
ylim = domain.dimensions[1][:2]
X, Y = np.meshgrid(np.linspace(*xlim, 257), np.linspace(*ylim, 257))
uXY = u_exact((X, Y))

fig, axs = plt.subplots(2, 2, subplot_kw=dict(projection="3d"))
fig.set_layout_engine("compressed")
fig.set_constrained_layout_pads(w_pad=0.25, h_pad=0.25)

for ax, res_offset in zip(axs.ravel(), range(0, 16, 4)):
    ax.plot_wireframe(
        X, Y, uXY, rstride=X.shape[0] - 1, cstride=X.shape[1] - 1, linewidths=0.5
    )

    for i, result in enumerate(results[res_offset : res_offset + 2]):
        num_divs = result["domain"].dimensions[0].num_divs
        linewidth = 4.0 / np.log2(num_divs)
        ax.plot_wireframe(
            *result["coords"],
            result["approx"],
            rstride=num_divs - 2,
            cstride=num_divs - 2,
            colors=f"C{i}",
            linewidths=linewidth,
        )
        ax.plot_wireframe(
            *result["coords"],
            result["approx"],
            rstride=1,
            cstride=1,
            colors=f"C{i}",
            alpha=0.5,
            linewidths=0.25 * linewidth,
        )

    ax.set(
        title=f"{result['ghost_term_bd']} {result["domain"].dimensions[0].high_robin}",
        xlim=xlim,
        ylim=ylim,
        xlabel="$x$",
        ylabel="$y$",
        zlabel="$u$",
    )

plt.show()

In [ ]:
for res_offset in range(0, 16, 4):
    result = results[res_offset]
    domain = result["domain"]
    xlim = domain.dimensions[0][:2]
    ylim = domain.dimensions[1][:2]

    fig, ax = plt.subplots(subplot_kw=dict(projection="3d"))

    for i, result in enumerate(results[res_offset : res_offset + 4]):
        num_divs = result["domain"].dimensions[0].num_divs
        linewidth = 4.0 / np.log2(num_divs)
        abserr = np.abs(result["error"])
        ax.plot_wireframe(
            *result["coords"],
            abserr,
            rstride=num_divs - 2,
            cstride=num_divs - 2,
            colors=f"C{i}",
            linewidths=linewidth,
            label=f"{num_divs}",
        )
        ax.plot_wireframe(
            *result["coords"],
            abserr,
            rstride=2,
            cstride=2,
            colors=f"C{i}",
            linewidths=0.5 * linewidth,
            alpha=0.5 * linewidth,
        )

    ax.set(
        title=f"{result['ghost_term_bd']} {result["domain"].dimensions[0].high_robin}",
        xlim=xlim,
        ylim=ylim,
        xlabel="$x$",
        ylabel="$y$",
        zlabel="|E|",
        zscale="log",
    )
    ax.legend()

plt.show()